# CS:GO Round Winner Classification
This notebook builds an end-to-end deep learning pipeline to predict the winner of a CS:GO round based on match snapshot data.

**Author:** Malik Tahayneh

In [2]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

## 1. Data Loading and Preprocessing
In this section, we load the raw `csgo_round_snapshots.csv` file, map our target variables to binary, and one-hot encode the categorical map names.

In [3]:
def load_dataset(file_path: str) -> pd.DataFrame:
    """
    Load the dataset into a pandas DataFrame.

    Args:
        file_path (str): The path to the 'csgo_round_snapshots.csv' file.

    Returns:
        pd.DataFrame: The loaded raw data.
    """
    df = pd.read_csv(file_path)
    return df

def clean_and_encode_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset and handle categorical variables.

    Tasks:
    1. Map boolean/text binary values (e.g., CT/T) to 1 and 0.
    2. Apply one-hot encoding for categorical variables like map names.
    3. Handle any missing or infinite values.

    Args:
        df (pd.DataFrame): The raw dataframe.

    Returns:
        pd.DataFrame: The numerically encoded dataframe.
    """
    df["round_winner"] = df["round_winner"].map({"CT": 1, "T":0})
    df = pd.get_dummies(df, columns=["map"], dtype=int)
    df = df.dropna()
    return df

# Execution
file_path = "csgo_round_snapshots.csv"
raw_df = load_dataset(file_path)
encoded_df = clean_and_encode_data(raw_df)

# Display the first 5 rows to visually verify the cleaning step
encoded_df.head()

,time_left,ct_score,t_score,bomb_planted,ct_health,t_health,ct_armor,t_armor,ct_money,t_money,...,t_grenade_decoygrenade,round_winner,map_de_cache,map_de_dust2,map_de_inferno,map_de_mirage,map_de_nuke,map_de_overpass,map_de_train,map_de_vertigo
0,175.00,0.0,0.0,False,500.0,500.0,0.0,0.0,4000.0,4000.0,...,0.0,1,0,1,0,0,0,0,0,0
1,156.03,0.0,0.0,False,500.0,500.0,400.0,300.0,600.0,650.0,...,0.0,1,0,1,0,0,0,0,0,0
2,96.03,0.0,0.0,False,391.0,400.0,294.0,200.0,750.0,500.0,...,0.0,1,0,1,0,0,0,0,0,0
3,76.03,0.0,0.0,False,391.0,400.0,294.0,200.0,750.0,500.0,...,0.0,1,0,1,0,0,0,0,0,0
4,174.97,1.0,0.0,False,500.0,500.0,192.0,0.0,18350.0,10750.0,...,0.0,1,0,1,0,0,0,0,0,0


## 2. Train/Test Split and Feature Scaling
Here we separate our target variable (`round_winner`), split the data into 80% training and 20% testing sets using stratification, and apply standard scaling to our features.

In [4]:
def split_data(df: pd.DataFrame, target_column: str):
    """
    Separate features (X) from the target (y) and split into training and testing sets.
    """
    y = df[target_column]
    X = df.drop(columns=[target_column], axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    return X_train, X_test, y_train, y_test

def scale_features(X_train, X_test):
    """
    Scale the features so the neural network can process them efficiently.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Execution
X_train, X_test, y_train, y_test = split_data(encoded_df, "round_winner")
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

print(f"Training features shape: {X_train_scaled.shape}")
print(f"Testing features shape: {X_test_scaled.shape}")

Training features shape: (97928, 103)
Testing features shape: (24482, 103)


## 3. Neural Network Architecture and Training
We build a 3-layer feed-forward neural network to output raw logits, optimized with Adam (learning rate 0.003) and Binary Crossentropy.

In [5]:
def train_neural_network(X_train_scaled, y_train):
    """
    Build, compile, and train the TensorFlow neural network.
    """
    model = Sequential([
        Dense(128, activation="relu", input_shape=(103, ), name="L1"),
        Dense(units=64, activation="relu", name="L2"),
        Dense(units=1, activation="linear", name="L3")
    ])

    model.compile(optimizer=Adam(learning_rate=0.002),
                  loss=BinaryCrossentropy(from_logits=True),
                  metrics=["accuracy", Precision(), Recall()])

    history = model.fit(
        x=X_train_scaled,
        y=y_train,
        epochs=50,
        batch_size=256,
        validation_split=0.2
    )
    return model, history

# Execution
model, history = train_neural_network(X_train_scaled, y_train)

Epoch 1/50


C:\Users\USER\PycharmProjects\DataScienceProgramming\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7296 - loss: 0.4708 - precision: 0.8242 - recall: 0.5697 - val_accuracy: 0.7503 - val_loss: 0.4489 - val_precision: 0.8232 - val_recall: 0.6255
Epoch 2/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7502 - loss: 0.4394 - precision: 0.8452 - recall: 0.6001 - val_accuracy: 0.7537 - val_loss: 0.4412 - val_precision: 0.8370 - val_recall: 0.6185
Epoch 3/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7606 - loss: 0.4253 - precision: 0.8470 - recall: 0.6243 - val_accuracy: 0.7453 - val_loss: 0.4325 - val_precision: 0.8711 - val_recall: 0.5645
Epoch 4/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7700 - loss: 0.4147 - precision: 0.8561 - recall: 0.6378 - val_accuracy: 0.7628 - val_loss: 0.4325 - val_precision: 0.8372 - val_recall: 0.6414
Epoch 5/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7781 - loss: 0.4062 - precision: 0.8546 - recall: 0.6593 - val_accuracy: 0.7708 - val_loss: 0.4220 - val_

In [6]:
def evaluate_model(model, X_test_scaled, y_test) -> None:
    """
    Evaluate the neural network's performance on the unseen test data.
    """
    raw_predictions = model.predict(X_test_scaled)
    y_pred = (raw_predictions > 0).astype(int)

    print("--- Classification Report ---")
    print(classification_report(y_test, y_pred, target_names=['CT Win', 'T Win']))
    print("\n--- Confusion Matrix ---")
    print(confusion_matrix(y_test, y_pred))

# Execution
evaluate_model(model, X_test_scaled, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 0s 354us/step
--- Classification Report ---
              precision    recall  f1-score   support

      CT Win       0.85      0.83      0.84     12481
       T Win       0.83      0.85      0.84     12001

    accuracy                           0.84     24482
   macro avg       0.84      0.84      0.84     24482
weighted avg       0.84      0.84      0.84     24482


--- Confusion Matrix ---
[[10332  2149]
 [ 1824 10177]]
